In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, confusion_matrix, roc_curve
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import matplotlib.pyplot as plt

BASE = Path(r"C:\Users\KIIT\Desktop\Parkinson_EEG")
RES = BASE/"results1"/"SanDiego"/"features_extracted"
OUT = BASE/"results1"/"SanDiego"/"case_results"
FIG = OUT/"figures"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

cases = {
    "ICA": RES/"caseA_ICA_features.npz",
    "Bandpass": RES/"caseB_Bandpass_features.npz"
}

models = {
    "KNN": KNeighborsClassifier(n_neighbors=7, weights="distance"),
    "SVM": SVC(C=1.0, kernel="rbf", probability=True),
    "RF": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, eval_metric="logloss", use_label_encoder=False),
    "LightGBM": LGBMClassifier(n_estimators=200),
    "CatBoost": CatBoostClassifier(iterations=200, verbose=0)
}

def run_case(tag, path):
    print(f"\n=== Training Case: {tag} ===")

    d = np.load(path, allow_pickle=True)
    X = d["X"]; y = d["y"]; subjects = d["subjects"]

    # -------- subject-wise 80/20 split --------
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    tr, te = next(gss.split(X, y, groups=subjects))
    Xtr, Xte = X[tr], X[te]
    ytr, yte = y[tr], y[te]
    sub_tr, sub_te = subjects[tr], subjects[te]

    scaler = StandardScaler().fit(Xtr)
    Xtr_s = scaler.transform(Xtr)
    Xte_s = scaler.transform(Xte)

    rows = []

    for name, model in models.items():
        m = model.fit(Xtr_s, ytr)

        ypred = m.predict(Xte_s)
        probs = m.predict_proba(Xte_s)[:,1] if hasattr(m,"predict_proba") else None

        acc = accuracy_score(yte, ypred)
        prec, rec, f1, _ = precision_recall_fscore_support(
            yte, ypred, average="binary", zero_division=0)

        cm = confusion_matrix(yte, ypred)
        tn, fp, fn, tp = cm.ravel()

        spec = tn / (tn+fp) if (tn+fp)>0 else 0
        npv  = tn / (tn+fn) if (tn+fn)>0 else 0
        auc  = roc_auc_score(yte, probs) if probs is not None else np.nan

        rows.append([name, acc, rec, spec, prec, npv, f1, auc])

        # ----- save confusion matrix -----
        plt.figure(figsize=(3.6,3))
        plt.imshow(cm, cmap="Blues")
        plt.title(f"{tag} — {name} (CM)")
        plt.xticks([0,1], ["HC","PD"])
        plt.yticks([0,1], ["HC","PD"])
        for i in range(2):
            for j in range(2):
                plt.text(j,i,cm[i,j],ha="center",va="center")
        plt.tight_layout()
        plt.savefig(FIG/f"cm_{tag}_{name}.png", dpi=200)
        plt.close()

        # ----- ROC -----
        if probs is not None:
            fpr,tpr,_ = roc_curve(yte, probs)
            plt.plot(fpr,tpr,label=f"{name} (AUC={auc:.3f})")

    # save ROC figure per case
    plt.plot([0,1],[0,1],"k--")
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title(f"ROC — {tag}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG/f"roc_{tag}.png", dpi=200)
    plt.close()

    df = pd.DataFrame(rows, columns=[
        "Model","Accuracy","Sensitivity","Specificity",
        "PPV","NPV","F1","AUC"
    ])

    out_csv = OUT/f"table2_{tag}_subjectwise_80_20.csv"
    df.to_csv(out_csv, index=False)
    print("Saved ->", out_csv)
    print(df)
    return df

df_ica = run_case("ICA_only", cases["ICA"])
df_band = run_case("Bandpass_only", cases["Bandpass"])


=== Training Case: ICA_only ===


C:\Users\KIIT\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [16:45:05] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[LightGBM] [Info] Number of positive: 2393, number of negative: 1151
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001162 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1785
[LightGBM] [Info] Number of data points in the train set: 3544, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.675226 -> initscore=0.731917
[LightGBM] [Info] Start training from score 0.731917
Saved -> C:\Users\KIIT\Desktop\Parkinson_EEG\results1\SanDiego\case_results\table2_ICA_only_subjectwise_80_20.csv
      Model  Accuracy  Sensitivity  Specificity       PPV       NPV        F1  \
0       KNN  0.685773     0.664441     0.719577  0.789683  0.575053  0.721668   
1       SVM  0.658137     0.993322     0.126984  0.643243  0.923077  0.780840   
2        RF  0.710338     0.756260     0.637566  0.767797  0.622739  0.761985   
3   XGBoos

C:\Users\KIIT\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [16:46:00] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[LightGBM] [Info] Number of positive: 4797, number of negative: 2308
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000881 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1785
[LightGBM] [Info] Number of data points in the train set: 7105, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.675158 -> initscore=0.731609
[LightGBM] [Info] Start training from score 0.731609
Saved -> C:\Users\KIIT\Desktop\Parkinson_EEG\results1\SanDiego\case_results\table2_Bandpass_only_subjectwise_80_20.csv
      Model  Accuracy  Sensitivity  Specificity       PPV       NPV        F1  \
0       KNN  0.651353     0.862729     0.315720  0.666881  0.591584  0.752267   
1       SVM  0.620725     0.995840     0.025099  0.618605  0.791667  0.763150   
2        RF  0.643696     0.977537     0.113606  0.636511  0.761062  0.770997   
3   X